# Flux Relighting Example

This notebook demonstrates a simple, yet effective way to produce aesthetic, in-distribution results, as described in section 5.1.1 of the [thesis](https://scholarsarchive.byu.edu/cgi/viewcontent.cgi?article=12256&context=etd#page=39.08). It works by first generating an image with FLUX that is conditioned on the base, unoptimized lighting for the scene (using the default values of all lights and no learned multipliers).

Then, using some metric like MSE, LPIPS, or SSIM, optimize the lighting multipliers to minimize the difference between the relighted image and the generated target image.

NOTE: To effectively use this notebook, you must uncomment the lines under `# Uncomment these for flux_relight_example.ipynb:` in `requirements.txt` and re-run the `pip install -r requirements.txt` command.

In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from losses.flux_relight_loss import (
    FLUXKontextRelighter,
    FluxLoss,
    RelightImageCache,
)

from examples.example_scenes import BlenderManScene, CandleScene, CarScene, CarStudioScene, DinoScene, EinarScene, EinarSmallDomeScene, FlowerPotScene, HouseScene, RedCarScene, SciFiRobotScene, SpringPortraitScene, SpringPortraitSmallDomeScene, SpringScene
from losses.image_image import (  # Though we could also use VGGStyleTransferLoss and ImageImbeddingSimilarityLoss, these image-image losses are better for directly comparing images that have nearly the same structural similarity:
    L1LossWithReferenceImage,
    LPIPSLoss,
    MSELossWithReferenceImage,
    SSIMLoss,
)
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.optimize import optimize_with_criterion
from utils.image.display import display_tensor


If the scene is constructed by including the alpha mask, then all non-white pixels in the alpha mask will be ignored when performing the optimization. This is useful because generally the background has more pixels than the foreground, inhibiting this method's ability to effectively optimize lighting on the subject.

In [ ]:
# Select the scene to optimize (uncomment the desired scene)
scene = SciFiRobotScene(include_alpha_mask=True, device=device)
# scene = SpringScene(device=device)
# scene = CarScene(device=device)
# scene = BlenderManScene(device=device)
# scene = RedCarScene(include_alpha_mask=True, device=device)
# scene = CandleScene(include_alpha_mask=True, device=device)
# scene = HouseScene(device=device)
# scene = DinoScene(device=device)
# scene = FlowerPotScene(device=device)
# scene = CarStudioScene(configuration='dome_lights', device=device)
# scene = EinarScene(device=device)
# scene = SpringPortraitScene(device=device)


In [ ]:
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

Optionally, display the initial scene with the chosen color space converter to visualize the starting point of the optimization:

In [ ]:
display_tensor(color_space_converter(scene.get_combined_image(None).permute(2, 1, 0)))

In [ ]:
# Hyperparameters
lr = 0.06
n_iter = 120
global_seed = 1

In [ ]:
relighter = FLUXKontextRelighter(seed=global_seed)
images_cache = RelightImageCache(relighter, image_to_relight=scene.get_combined_image(color_space_converter).permute(2, 0, 1).unsqueeze(0).to(device))

In [ ]:
target_text = "aesthetic, golden hour lighting"
num_results = 4

In [ ]:
criterion_mse = FluxLoss(cache=images_cache, image_comparison_criterion_cls=MSELossWithReferenceImage, target_text=target_text, num_relighted_images=num_results, display=True)
# criterion_lpips = FluxLoss(cache=images_cache, image_comparison_criterion_cls=LPIPSLoss, target_text=target_text, num_relighted_images=num_results, display=True)
# criterion_ssim = FluxLoss(cache=images_cache, image_comparison_criterion_cls=SSIMLoss, target_text=target_text, num_relighted_images=num_results, display=True)
# criterion_l1 = FluxLoss(cache=images_cache, image_comparison_criterion_cls=L1LossWithReferenceImage, target_text=target_text, num_relighted_images=num_results, display=True)

for criterion, title in [
    (criterion_mse, "MSE"),
    # (criterion_lpips, "LPIPS"),
    # (criterion_ssim, "SSIM"),
    # (criterion_l1, "L1")
    ]:

    title_prefix = f"Flux Relight ({target_text}) with {title} Loss"

    optimize_with_criterion(
        scene,
        lr,
        n_iter,
        criterion,
        starting_multiplier_std=(0.1, 0.1, 0.1),
        output_subdirectory_name="flux_relight_example",
        n_results=1,
        render_color_space_converter=color_space_converter,
        require_physically_plausible_multipliers=True,
        title_prefix=title_prefix,
        device=device,
        save_every=10,
        model_name="flux",
        pretrained_source="flux",
        seed=global_seed,
        show_images_after_augmentation=False,
    )
